# Monte Carlo Estimator Validation & Geometric Confound Analysis

This notebook demonstrates:

- **Task A (Monte Carlo):** Simulation of AR(1) sequences comparing 3 lag-1 autocorrelation estimators (standard Yule-Walker, Marriott-Pope corrected, profile MLE) across parameter cells defined by sequence length and true phi. Computes bias, RMSE, coverage, power, minimum detectable effects (MDE), and bias-cancellation verification.

- **Task B (Geometric Confounds):** Three-pronged analysis on synthetic canonical trees (star, caterpillar, balanced, random projective) and real UD dependency sentences to determine whether tree geometry alone explains observed anti-correlation, using Random Projective Linearisation (RPL) baselines.

**Data:** Mini demo dataset of UD dependency tree sentences for Task B real-tree analysis. Task A is fully self-contained (generates synthetic AR(1) sequences).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-Colab packages needed — all imports are in the core scientific stack

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    # Colab versions: numpy==2.0.2, scipy==1.16.3, matplotlib==3.10.0
    # scipy>=1.16 requires Python 3.11+; use compatible versions for 3.10
    if sys.version_info >= (3, 11):
        _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')
    else:
        _pip('numpy==2.0.2', 'scipy==1.15.3', 'matplotlib==3.10.0')

In [ ]:
import hashlib
import itertools
import json
import os
import time
from collections import defaultdict

import numpy as np
import scipy.stats
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-c3cfa4-sequential-dependency-distance-anti-corr/main/experiment_iter3_mc_geometry/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data.get('datasets', []))} dataset(s)")
for ds in data.get('datasets', []):
    print(f"  {ds.get('dataset', '?')}: {len(ds.get('examples', []))} examples")

## Configuration

All tunable parameters for both tasks. Values below are set for a quick demo run.
Commented values show the original full-scale parameters used in the paper.

In [ ]:
# ── Task A: Monte Carlo simulation parameters ──
LENGTHS = [10, 20]                    # Original: [10, 15, 20, 25, 30, 40]
PHIS = np.array([-0.2, 0.0, 0.2])    # Original: np.round(np.arange(-0.40, 0.45, 0.05), 2)  (17 values)
N_REPS = 100                          # Original: 10_000
MLE_SUBSAMPLE = 10                    # Original: 200
CELLS = list(itertools.product(LENGTHS, [float(p) for p in PHIS]))

# Bias-cancellation test
BIAS_CANCEL_CELLS = [(20, 0.0)]       # Original: [(20,-0.20),(20,-0.10),(20,0.0),(20,0.10),(30,-0.20),(30,0.0)]
N_PAIRS = 20                          # Original: 1000
N_BASELINE_SAMPLES = 10               # Original: 50

# ── Task B: Geometric confound parameters ──
TREE_SIZES = [20]                     # Original: [20, 30, 40]
N_TREES_SYNTHETIC = 5                 # Original: 500
N_RPL_PERMS = 10                      # Original: 100
MAX_SENTENCES_REAL = 9                # Original: 2000
MIN_TOKENS = 15                       # same as original

print(f"Task A: {len(CELLS)} cells, {N_REPS} reps/cell, MLE sub={MLE_SUBSAMPLE}")
print(f"Task B: {len(TREE_SIZES)} tree sizes, {N_TREES_SYNTHETIC} trees, {N_RPL_PERMS} RPL perms")

## Helper Functions

Utility functions for seeding, type conversion, and autocorrelation estimators.

In [ ]:
def make_seed(*args) -> int:
    """Deterministic seed from arbitrary arguments."""
    h = hashlib.sha256(str(args).encode()).hexdigest()
    return int(h[:8], 16) % (2**31)


def to_native(obj):
    """Recursively convert numpy types to Python builtins for JSON."""
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_native(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


# ── Autocorrelation Estimators ──

def r1_standard_batch(X: np.ndarray) -> np.ndarray:
    """Vectorised standard (Yule-Walker) lag-1 autocorrelation per row."""
    Xc = X - X.mean(axis=1, keepdims=True)
    num = np.sum(Xc[:, :-1] * Xc[:, 1:], axis=1)
    den = np.sum(Xc ** 2, axis=1)
    den = np.where(den == 0, 1e-30, den)
    return num / den


def r1_corrected_batch(X: np.ndarray) -> np.ndarray:
    """Marriott-Pope analytical correction: r1 + (1 + 3*r1) / n."""
    r1 = r1_standard_batch(X)
    n = X.shape[1]
    return r1 + (1.0 + 3.0 * r1) / n


def compute_r1(seq) -> float:
    """Standard lag-1 autocorrelation of a 1-D numeric sequence."""
    x = np.asarray(seq, dtype=float)
    if len(x) < 4:
        return np.nan
    xc = x - x.mean()
    den = np.dot(xc, xc)
    if den == 0:
        return 0.0
    return float(np.dot(xc[:-1], xc[1:]) / den)


def estimator_mle_scipy(x: np.ndarray) -> float:
    """Profile-MLE for AR(1) phi via scipy bounded minimisation."""
    n = len(x)
    if n < 4:
        return np.nan
    y = x - np.mean(x)
    y0sq = y[0] ** 2
    y_lag = y[:-1]
    y_lead = y[1:]

    def neg_prof_ll(phi):
        if abs(phi) >= 0.999:
            return 1e10
        resid = y_lead - phi * y_lag
        ssr = (1 - phi ** 2) * y0sq + np.dot(resid, resid)
        if ssr <= 0:
            return 1e10
        return n * np.log(ssr / n) - np.log(1 - phi ** 2)

    try:
        res = minimize_scalar(neg_prof_ll, bounds=(-0.99, 0.99),
                              method="bounded")
        return float(res.x) if res.success else np.nan
    except Exception:
        return np.nan


# ── AR(1) Sequence Generation ──

def generate_ar1_batch(n: int, phi: float, n_reps: int,
                       rng: np.random.Generator) -> np.ndarray:
    """Generate n_reps AR(1) sequences of length n (rows of a 2-D array)."""
    eps = rng.normal(0.0, 1.0, size=(n_reps, n))
    x = np.empty_like(eps)
    if abs(phi) < 0.999:
        x[:, 0] = eps[:, 0] / np.sqrt(1 - phi ** 2)
    else:
        x[:, 0] = eps[:, 0]
    for t in range(1, n):
        x[:, t] = phi * x[:, t - 1] + eps[:, t]
    return x

print("Helper functions defined.")

## Task A: Monte Carlo Estimator Validation

Simulate AR(1) sequences for each (length, phi) cell and compute bias, RMSE, coverage, and power for all three estimators. Also builds the MDE table and runs bias-cancellation verification.

In [ ]:
def process_cell(args):
    """Compute bias / RMSE / coverage / power for one (n, phi) cell."""
    n, phi, seed, n_reps, mle_sub = args
    rng = np.random.default_rng(seed)
    X = generate_ar1_batch(n, phi, n_reps, rng)

    # Vectorised estimators
    r1_vals = r1_standard_batch(X)
    r1c_vals = r1_corrected_batch(X)

    # MLE on a subsample
    mle_sub_actual = min(mle_sub, n_reps)
    mle_vals = np.array([estimator_mle_scipy(X[i])
                         for i in range(mle_sub_actual)])
    mle_vals = mle_vals[~np.isnan(mle_vals)]

    se_null = 1.0 / np.sqrt(n)
    results = {}

    for name, vals in [("r1_standard", r1_vals),
                       ("r1_corrected", r1c_vals),
                       ("mle", mle_vals)]:
        bias  = float(np.mean(vals) - phi)
        rmse  = float(np.sqrt(np.mean((vals - phi) ** 2)))
        se_est = float(np.std(vals))
        ci_lo = vals - 1.96 * se_est
        ci_hi = vals + 1.96 * se_est
        coverage = float(np.mean((ci_lo <= phi) & (phi <= ci_hi)))
        z = np.abs(vals) / se_null
        power = float(np.mean(z > 1.96))

        results[name] = {
            "bias":          bias,
            "rmse":          rmse,
            "coverage_95":   coverage,
            "power_alpha05": power,
            "mean_estimate": float(np.mean(vals)),
            "std_estimate":  se_est,
            "n_valid":       int(len(vals)),
        }

    return (n, float(phi), results)


# ── Run Task A ──
t0 = time.time()
print(f"Task A: {len(LENGTHS)} lengths x {len(PHIS)} phi = {len(CELLS)} cells, "
      f"{N_REPS} reps/cell, MLE sub-sample {MLE_SUBSAMPLE}")

base_rng   = np.random.default_rng(42)
cell_seeds = base_rng.integers(0, 2**31, size=len(CELLS)).tolist()
cell_args  = [(n, float(phi), seed, N_REPS, MLE_SUBSAMPLE)
              for (n, phi), seed in zip(CELLS, cell_seeds)]

mc_results = {}
for a in cell_args:
    n_val, phi_val, res = process_cell(a)
    mc_results[f"n{n_val}_phi{phi_val:.2f}"] = res

ta_time = time.time() - t0
print(f"Task A done in {ta_time:.1f}s ({len(mc_results)} cells)")

# ── MDE table ──
mde_table = {}
for n_val in LENGTHS:
    for est in ("r1_standard", "r1_corrected", "mle"):
        mde = None
        for pv in sorted(float(p) for p in PHIS if p > 0):
            key = f"n{n_val}_phi{pv:.2f}"
            if (key in mc_results
                    and mc_results[key][est]["power_alpha05"] >= 0.80):
                mde = pv
                break
        mde_table[f"n{n_val}_{est}"] = mde
print(f"MDE table: {json.dumps(to_native(mde_table), indent=2)}")

# ── Bias-cancellation test ──
print("Bias-cancellation test ...")
bias_cancel = {}
for n_val, phi_val in BIAS_CANCEL_CELLS:
    rng = np.random.default_rng(make_seed("bias_cancel", n_val, phi_val))
    X_A  = generate_ar1_batch(n_val, phi_val, N_PAIRS, rng)
    r1_A = r1_standard_batch(X_A)
    exc  = np.empty(N_PAIRS)
    for i in range(N_PAIRS):
        X_B = generate_ar1_batch(n_val, phi_val, N_BASELINE_SAMPLES, rng)
        exc[i] = r1_A[i] - float(np.mean(r1_standard_batch(X_B)))
    k = f"n{n_val}_phi{phi_val:.2f}"
    lo, hi = float(np.percentile(exc, 2.5)), float(np.percentile(exc, 97.5))
    bias_cancel[k] = {
        "mean_excess":        float(np.mean(exc)),
        "std_excess":         float(np.std(exc)),
        "ci_95_low":          lo,
        "ci_95_high":         hi,
        "proportion_negative": float(np.mean(exc < 0)),
        "zero_in_ci":         bool(lo <= 0 <= hi),
    }
    print(f"  {k}: mean={bias_cancel[k]['mean_excess']:.6f}  "
          f"zero_in_ci={bias_cancel[k]['zero_in_ci']}")

## Task B: Geometric Confound Analysis

Tree utility functions and synthetic/real tree generators for the three-pronged geometric confound check.

In [ ]:
# ── Tree Utilities ──

def build_children_map(head_array):
    """Return (children dict, root index). head_array is 1-indexed; 0 means root."""
    children = defaultdict(list)
    root = None
    for i, h in enumerate(head_array):
        if h == 0:
            root = i
        else:
            children[h - 1].append(i)
    return children, root


def rpl_linearize(children, node, rng):
    """Random Projective Linearisation -- Gildea-Temperley method."""
    kids = children.get(node, [])
    if not kids:
        return [node]
    left, right = [], []
    for k in kids:
        (left if rng.random() < 0.5 else right).append(k)
    rng.shuffle(left)
    rng.shuffle(right)
    out = []
    for k in left:
        out.extend(rpl_linearize(children, k, rng))
    out.append(node)
    for k in right:
        out.extend(rpl_linearize(children, k, rng))
    return out


def compute_dd_from_linearization(linearization, head_array):
    """DD sequence from a linearisation (excludes root token)."""
    pos_map = {orig: new + 1 for new, orig in enumerate(linearization)}
    dd = []
    for new_idx, orig in enumerate(linearization):
        h = head_array[orig]
        if h == 0:
            continue
        head_orig = h - 1
        if head_orig in pos_map:
            dd.append(abs((new_idx + 1) - pos_map[head_orig]))
    return dd


def compute_dd_filtered(linearization, head_array, deprel_array):
    """DD sequence excluding root and punct tokens."""
    pos_map = {orig: new + 1 for new, orig in enumerate(linearization)}
    dd = []
    for new_idx, orig in enumerate(linearization):
        h = head_array[orig]
        if h == 0:
            continue
        if deprel_array[orig] == "punct":
            continue
        head_orig = h - 1
        if head_orig in pos_map:
            dd.append(abs((new_idx + 1) - pos_map[head_orig]))
    return dd


# ── Synthetic Tree Generators ──

def make_star_tree(n):
    head = [0] * n
    for i in range(1, n):
        head[i] = 1
    return head

def make_caterpillar_tree(n):
    head = [0] * n
    spine = n // 2
    for i in range(1, spine):
        head[i] = i
    leaf = spine
    for i in range(spine):
        if leaf < n:
            head[leaf] = i + 1
            leaf += 1
    return head

def make_balanced_tree(n):
    head = [0] * n
    for i in range(1, n):
        head[i] = (i - 1) // 2 + 1
    return head

def make_random_projective_tree(n, rng):
    head = [0] * n
    def _build(positions, parent_pos):
        if not positions:
            return
        ri = int(rng.integers(0, len(positions)))
        rp = positions[ri]
        if parent_pos is not None:
            head[rp] = parent_pos + 1
        left  = [p for p in positions if p < rp]
        right = [p for p in positions if p > rp]
        _build(left, rp)
        _build(right, rp)
    _build(list(range(n)), None)
    return head

print("Tree utilities defined.")

### Prong 1: Synthetic Canonical Trees

Compute excess autocorrelation (canonical vs RPL baseline) for four tree topologies at each tree size.

In [ ]:
import sys
sys.setrecursionlimit(5000)

print("Prong 1 -- synthetic trees ...")
synthetic_results = {}
tree_gens = [
    ("star",              make_star_tree,              False),
    ("caterpillar",       make_caterpillar_tree,       False),
    ("balanced",          make_balanced_tree,          False),
    ("random_projective", make_random_projective_tree, True),
]

for ttype, gen_fn, needs_rng in tree_gens:
    for sz in TREE_SIZES:
        excesses = []
        obs_r1s  = []
        rpl_means = []

        for t_idx in range(N_TREES_SYNTHETIC):
            rng = np.random.default_rng(make_seed(ttype, sz, t_idx))
            harr = gen_fn(sz, rng) if needs_rng else gen_fn(sz)

            children, root = build_children_map(harr)
            if root is None:
                continue

            # Canonical (natural position) linearisation
            dd_can = compute_dd_from_linearization(list(range(sz)), harr)
            r1_can = compute_r1(dd_can)

            # RPL permutations
            rpl_r1_list = []
            for _ in range(N_RPL_PERMS):
                try:
                    lin = rpl_linearize(children, root, rng)
                    r1v = compute_r1(compute_dd_from_linearization(lin, harr))
                    if not np.isnan(r1v):
                        rpl_r1_list.append(r1v)
                except RecursionError:
                    continue

            if rpl_r1_list and not np.isnan(r1_can):
                ex = r1_can - float(np.mean(rpl_r1_list))
                excesses.append(ex)
                obs_r1s.append(r1_can)
                rpl_means.append(float(np.mean(rpl_r1_list)))

        key = f"{ttype}_n{sz}"
        if excesses:
            ea = np.array(excesses)
            synthetic_results[key] = {
                "mean_excess":        float(np.mean(ea)),
                "std_excess":         float(np.std(ea)),
                "median_excess":      float(np.median(ea)),
                "mean_obs_r1":        float(np.mean(obs_r1s)),
                "mean_rpl_r1":        float(np.mean(rpl_means)),
                "n_trees_computed":   len(excesses),
                "proportion_negative": float(np.mean(ea < 0)),
            }
        else:
            synthetic_results[key] = {
                "mean_excess": 0.0, "std_excess": 0.0,
                "median_excess": 0.0, "mean_obs_r1": 0.0,
                "mean_rpl_r1": 0.0, "n_trees_computed": 0,
                "proportion_negative": 0.0,
            }
        print(f"  {key}: mean_excess={synthetic_results[key]['mean_excess']:.4f}  "
              f"n={synthetic_results[key]['n_trees_computed']}")

### Prong 2-3: Real UD Dependency Trees

Load sentences from mini demo data, then compute observed-vs-RPL, RPL-vs-RPL, and shuffled-vs-RPL excess autocorrelation to test whether sequential effects are genuine or explained by tree geometry alone.

In [ ]:
# ── Extract sentences from loaded data ──
def _extract_sentences(data_dict, min_tokens):
    """Pull qualifying sentences from the loaded demo data."""
    sents = []
    for ds in data_dict.get("datasets", []):
        for ex in ds.get("examples", []):
            try:
                inp = json.loads(ex["input"]) if isinstance(ex["input"], str) else ex["input"]
                head  = inp["head_array"]
                deprl = inp["deprel_array"]
                keep  = [i for i, d in enumerate(deprl) if d != "punct"]
                if len(keep) < min_tokens:
                    continue
                out_raw = ex.get("output")
                if isinstance(out_raw, str):
                    out = json.loads(out_raw)
                else:
                    out = out_raw or {}
                sents.append({
                    "head_array":   head,
                    "deprel_array": deprl,
                    "keep_idx":     keep,
                    "token_count":  out.get("token_count", len(head)),
                    "treebank_id":  ex.get("metadata_treebank_id",
                                           ds.get("dataset", "unknown")),
                })
            except (json.JSONDecodeError, KeyError):
                continue
    return sents

all_sents = _extract_sentences(data, MIN_TOKENS)
if len(all_sents) > MAX_SENTENCES_REAL:
    rng_sub = np.random.default_rng(123)
    idx = rng_sub.choice(len(all_sents), size=MAX_SENTENCES_REAL, replace=False)
    all_sents = [all_sents[i] for i in sorted(idx)]
print(f"Loaded {len(all_sents)} qualifying sentences for real-tree analysis")

# ── Prong 2 & 3 ──
print("Prong 2-3 -- real UD trees ...")
rpl_vs_rpl_exc  = []
obs_vs_rpl_exc  = []
shuf_vs_rpl_exc = []

n_rpl_real = min(N_RPL_PERMS * 10, 100)  # use more RPL perms for real trees (need >= 50)

tb_start = time.time()
for s_idx, sent in enumerate(all_sents):
    harr  = sent["head_array"]
    drels = sent["deprel_array"]
    children, root = build_children_map(harr)
    if root is None:
        continue

    # Observed DD (punct-filtered, original word order)
    keep = sent["keep_idx"]
    obs_dd = [abs(i + 1 - harr[i]) for i in keep if harr[i] != 0]
    r1_obs = compute_r1(obs_dd)

    # Generate RPL permutations
    rng_r = np.random.default_rng(make_seed("rpl", s_idx))
    rpl_r1s = []
    for _ in range(n_rpl_real):
        try:
            lin = rpl_linearize(children, root, rng_r)
            dd_f = compute_dd_filtered(lin, harr, drels)
            val = compute_r1(dd_f)
            if not np.isnan(val):
                rpl_r1s.append(val)
        except RecursionError:
            continue

    if len(rpl_r1s) < 50:
        continue

    # Prong 2 -- RPL-vs-RPL geometric null
    h1 = float(np.mean(rpl_r1s[:50]))
    h2 = float(np.mean(rpl_r1s[50:]))
    rpl_vs_rpl_exc.append(h1 - h2)

    # Prong 3 -- observed vs RPL, shuffled vs RPL
    if not np.isnan(r1_obs):
        rpl_m = float(np.mean(rpl_r1s))
        obs_vs_rpl_exc.append(r1_obs - rpl_m)

        shuf_dd = list(rng_r.permutation(obs_dd))
        r1_shuf = compute_r1(shuf_dd)
        if not np.isnan(r1_shuf):
            shuf_vs_rpl_exc.append(r1_shuf - rpl_m)

tb_time = time.time() - tb_start
print(f"Real-tree results ({tb_time:.1f}s): "
      f"RPL-vs-RPL={len(rpl_vs_rpl_exc)}, "
      f"obs-vs-RPL={len(obs_vs_rpl_exc)}, "
      f"shuf-vs-RPL={len(shuf_vs_rpl_exc)}")

# ── Assemble geometric results ──
geo_results = {}

if len(rpl_vs_rpl_exc) > 1:
    geo_results["rpl_vs_rpl"] = {
        "mean_excess":  float(np.mean(rpl_vs_rpl_exc)),
        "std_excess":   float(np.std(rpl_vs_rpl_exc)),
        "p_value_ttest": float(scipy.stats.ttest_1samp(rpl_vs_rpl_exc, 0).pvalue),
        "n_sentences":  len(rpl_vs_rpl_exc),
        "near_zero":    bool(abs(np.mean(rpl_vs_rpl_exc)) < 0.01),
    }

if len(obs_vs_rpl_exc) > 1:
    geo_results["observed_vs_rpl"] = {
        "mean_excess":        float(np.mean(obs_vs_rpl_exc)),
        "std_excess":         float(np.std(obs_vs_rpl_exc)),
        "p_value_ttest":      float(scipy.stats.ttest_1samp(obs_vs_rpl_exc, 0).pvalue),
        "n_sentences":        len(obs_vs_rpl_exc),
        "median_excess":      float(np.median(obs_vs_rpl_exc)),
        "proportion_negative": float(np.mean(np.array(obs_vs_rpl_exc) < 0)),
    }

if len(shuf_vs_rpl_exc) > 1:
    geo_results["shuffled_vs_rpl"] = {
        "mean_excess":  float(np.mean(shuf_vs_rpl_exc)),
        "std_excess":   float(np.std(shuf_vs_rpl_exc)),
        "p_value_ttest": float(scipy.stats.ttest_1samp(shuf_vs_rpl_exc, 0).pvalue),
        "n_sentences":  len(shuf_vs_rpl_exc),
    }

if len(obs_vs_rpl_exc) > 1 and len(shuf_vs_rpl_exc) > 1:
    geo_results["observed_vs_shuffled_comparison"] = {
        "mean_diff":    float(np.mean(obs_vs_rpl_exc) - np.mean(shuf_vs_rpl_exc)),
        "ttest_pvalue": float(scipy.stats.ttest_ind(obs_vs_rpl_exc, shuf_vs_rpl_exc).pvalue),
    }

print(f"Geometric results keys: {list(geo_results.keys())}")

## Results Summary & Visualization

Print key metrics in a readable table and generate diagnostic plots for both tasks.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# RESULTS TABLE
# ═══════════════════════════════════════════════════════════════════

# ── Task A summary ──
print("=" * 70)
print("TASK A: Monte Carlo Estimator Results")
print("=" * 70)
print(f"{'Cell':<20} {'Estimator':<15} {'Bias':>8} {'RMSE':>8} {'Cover':>8} {'Power':>8}")
print("-" * 70)
for key in sorted(mc_results):
    for est in ("r1_standard", "r1_corrected", "mle"):
        r = mc_results[key][est]
        print(f"{key:<20} {est:<15} {r['bias']:>8.4f} {r['rmse']:>8.4f} "
              f"{r['coverage_95']:>8.3f} {r['power_alpha05']:>8.3f}")

print(f"\nMDE Table: {json.dumps(to_native(mde_table), indent=2)}")
print(f"\nBias Cancellation:")
for k, v in bias_cancel.items():
    print(f"  {k}: mean_excess={v['mean_excess']:.6f}, zero_in_ci={v['zero_in_ci']}")

# Interpret
bias_cancels = all(v["zero_in_ci"] for v in bias_cancel.values())
chosen_est = "r1_standard" if bias_cancels else "r1_corrected"
print(f"\nChosen estimator: {chosen_est} (bias cancels: {bias_cancels})")

# ── Task B summary ──
print("\n" + "=" * 70)
print("TASK B: Geometric Confound Results")
print("=" * 70)

print("\nSynthetic trees:")
print(f"{'Key':<25} {'Mean Excess':>12} {'Std':>10} {'N trees':>8} {'Prop Neg':>10}")
print("-" * 70)
for key in sorted(synthetic_results):
    r = synthetic_results[key]
    print(f"{key:<25} {r['mean_excess']:>12.4f} {r['std_excess']:>10.4f} "
          f"{r['n_trees_computed']:>8} {r['proportion_negative']:>10.3f}")

print("\nReal-tree geometric checks:")
for k, v in geo_results.items():
    print(f"  {k}: {json.dumps(to_native(v), indent=4)}")

# ═══════════════════════════════════════════════════════════════════
# PLOTS
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Bias curves ──
ax = axes[0]
for est, clr, ls in [("r1_standard", "#2196F3", "-"),
                      ("r1_corrected", "#FF9800", "--"),
                      ("mle", "#4CAF50", "-.")]:
    for n_val in LENGTHS:
        bi = [mc_results.get(f"n{n_val}_phi{p:.2f}", {}).get(est, {}).get("bias", np.nan)
              for p in PHIS]
        ax.plot(PHIS, bi, color=clr, ls=ls, marker=".", ms=5, lw=1.5,
                label=f"{est} (n={n_val})")
# Theoretical first-order bias for standard r1 (smallest n)
n_show = LENGTHS[0]
theo = [-(1 + 3 * float(p)) / n_show for p in PHIS]
ax.plot(PHIS, theo, color="gray", ls=":", alpha=.6, label=f"theo bias n={n_show}")
ax.axhline(0, color="black", ls="-", alpha=.3)
ax.set_title("Bias by Estimator & Length")
ax.set_xlabel("True phi"); ax.set_ylabel("Bias (E[est] - phi)")
ax.legend(fontsize=6, ncol=2); ax.grid(alpha=.3)

# ── Plot 2: Synthetic tree excess bar chart ──
ax = axes[1]
colours = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]
tree_types = ["star", "caterpillar", "balanced", "random_projective"]
x_pos = np.arange(len(TREE_SIZES))
width = 0.2
for i, ttype in enumerate(tree_types):
    means = [synthetic_results.get(f"{ttype}_n{s}", {}).get("mean_excess", 0) for s in TREE_SIZES]
    stds  = [synthetic_results.get(f"{ttype}_n{s}", {}).get("std_excess",  0) for s in TREE_SIZES]
    ax.bar(x_pos + i * width, means, width, yerr=stds, capsize=3,
           alpha=.7, color=colours[i], label=ttype)
ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels([str(s) for s in TREE_SIZES])
ax.axhline(0, color="red", ls="--", lw=1)
ax.set_title("Synthetic Tree Excess r1")
ax.set_xlabel("Tree Size"); ax.set_ylabel("Excess r1")
ax.legend(fontsize=7); ax.grid(alpha=.3, axis="y")

plt.tight_layout()
plt.show()

# ── Plot 3: Real-tree excess histograms (if data available) ──
if obs_vs_rpl_exc or rpl_vs_rpl_exc:
    fig, ax = plt.subplots(figsize=(10, 5))
    nbins = lambda lst: min(30, max(5, len(lst) // 3)) if lst else 5
    if rpl_vs_rpl_exc:
        ax.hist(rpl_vs_rpl_exc, bins=nbins(rpl_vs_rpl_exc), alpha=.5,
                density=True, color="#2196F3",
                label=f"RPL-vs-RPL (n={len(rpl_vs_rpl_exc)})")
    if obs_vs_rpl_exc:
        ax.hist(obs_vs_rpl_exc, bins=nbins(obs_vs_rpl_exc), alpha=.5,
                density=True, color="#F44336",
                label=f"Observed-vs-RPL (n={len(obs_vs_rpl_exc)})")
    if shuf_vs_rpl_exc:
        ax.hist(shuf_vs_rpl_exc, bins=nbins(shuf_vs_rpl_exc), alpha=.5,
                density=True, color="#4CAF50",
                label=f"Shuffled-vs-RPL (n={len(shuf_vs_rpl_exc)})")
    ax.axvline(0, color="black", ls="--", lw=1)
    ax.legend(fontsize=9)
    ax.set_xlabel("Excess lag-1 autocorrelation"); ax.set_ylabel("Density")
    ax.set_title("Geometric Confound Check: Real UD Sentences")
    ax.grid(alpha=.3)
    plt.tight_layout()
    plt.show()
else:
    print("(No real-tree data available for histogram)")

print(f"\nTotal runtime: {time.time() - t0:.1f}s")